In [40]:
import pandas as pd
import nltk

In [41]:
df = pd.read_csv("sentiment_analysis.csv")

In [42]:
df.head()

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           499 non-null    int64 
 1   Month          499 non-null    int64 
 2   Day            499 non-null    int64 
 3   Time of Tweet  499 non-null    object
 4   text           499 non-null    object
 5   sentiment      499 non-null    object
 6   Platform       499 non-null    object
dtypes: int64(3), object(4)
memory usage: 27.4+ KB


In [44]:
df.drop(columns =["Year","Month","Day","Time of Tweet","Platform"],axis = 1,inplace = True)

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499 entries, 0 to 498
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       499 non-null    object
 1   sentiment  499 non-null    object
dtypes: object(2)
memory usage: 7.9+ KB


In [46]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["sentiment"] = encoder.fit_transform(df["sentiment"])
y  = df["sentiment"]

In [47]:
import re
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]','',text)
    return text

df["clean_text"] = df["text"].apply(clean)

In [48]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word  not in stop_words]
    return " ".join(words)

In [49]:
df

,text,sentiment,clean_text
0,What a great day!!! Looks like dream.,2,whatagreatdaylookslikedream
1,"I feel sorry, I miss you here in the sea beach",2,ifeelsorryimissyouhereintheseabeach
2,Don't angry me,0,dontangryme
3,We attend in the class just for listening teac...,0,weattendintheclassjustforlisteningteachersread...
4,"Those who want to go, let them go",0,thosewhowanttogoletthemgo
...,...,...,...
494,"According to , a quarter of families under six...",0,accordingtoaquarteroffamiliesundersixliveinpov...
495,the plan to not spend money is not going well,0,theplantonotspendmoneyisnotgoingwell
496,uploading all my bamboozle pictures of facebook,1,uploadingallmybamboozlepicturesoffacebook
497,congratulations ! you guys finish a month ear...,2,congratulationsyouguysfinishamonthearlythanwed...


In [50]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(
    num_words = 5000
)
tokenizer.fit_on_texts(df["clean_text"])
X=tokenizer.texts_to_sequences(df["clean_text"])

In [51]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X = pad_sequences(X,maxlen=50,padding = "post")

In [52]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = .2,random_state = 42)

In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()
model.add(
    Embedding(5000,64)
)
model.add(
    LSTM(64)
)
model.add(
    Dense(
        3,activation = "softmax",
    )
)

In [54]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [55]:
model.fit(X_train,
          y_train,
          epochs = 20,
         batch_size= 32)

Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.3609 - loss: 1.0896
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4236 - loss: 1.0768
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4236 - loss: 1.0761
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4236 - loss: 1.0754
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4236 - loss: 1.0765
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4236 - loss: 1.0756
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4236 - loss: 1.0756
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4236 - loss: 1.0766
Epoch 9/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4236 - loss: 1.0773
Epoch 10/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4236 - loss: 1.0753
Epoch 11/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.4236 - loss: 1.0764
Epoch 12/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy:

In [64]:
import numpy as np
pred = model.predict(X_test)
pred = np.argmax(pred,1)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


## as dataset is very small so thats why we can make a efficient lstm rnn thats why metrics are poor

In [65]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, average="weighted"))
print("Recall:", recall_score(y_test, pred, average="weighted"))
print("F1 Score:", f1_score(y_test, pred, average="weighted"))

Accuracy: 0.3
Precision: 0.09
Recall: 0.3
F1 Score: 0.13846153846153847


C:\Users\manjo\Desktop\sample_project\env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
